In [1]:
# Fix Python path to include user-installed packages for BOTH Python 3.10 and 3.12
# import sys
# import importlib
# import site


# Note: NumPy Compatibility Warning

If you see a NumPy 1.x vs 2.x warning, you can safely ignore it. The code works fine despite the warning.

To fix it permanently, you would need to downgrade NumPy, but this requires disk space:
```python
# pip install "numpy<2" --user
```

In [2]:
# Import Earth Engine
import ee
import geemap

# Authenticate (only needed first time)
ee.Authenticate()

# Initialize Earth Engine
ee.Initialize(project="ee-ktwu01")

print("Earth Engine initialized successfully!")

Earth Engine initialized successfully!


In [3]:
# Load collection
dataset = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")

# Point of interest
point = ee.Geometry.Point(-121.8036, 39.0372)

# Get embedding images for two years
image1 = dataset.filterDate("2023-01-01", "2024-01-01").filterBounds(point).first()

image2 = dataset.filterDate("2024-01-01", "2025-01-01").filterBounds(point).first()

# Visualization parameters
vis_params = {"min": -0.3, "max": 0.3, "bands": ["A01", "A16", "A09"]}

# Calculate dot product (similarity measure)
dot_prod = image1.multiply(image2).reduce(ee.Reducer.sum())


# Print out some information about the images
def print_image_details():
    print("<b>2023 Image Details:</b>")
    print(f"Bands: {image1.bandNames().getInfo()}")
    print(f"Image Projection: {image1.projection().getInfo()}")

    print("\n<b>2024 Image Details:</b>")
    print(f"Bands: {image2.bandNames().getInfo()}")
    print(f"Image Projection: {image2.projection().getInfo()}")

    print("\n<b>Dot Product Similarity:</b>")
    print(f"Similarity Value: {dot_prod.getInfo()}")


# Run the detailed analysis
print_image_details()

# Optional: Export images
# Uncomment the following line if you want to export images
# export_images()

<b>2023 Image Details:</b>
Bands: ['A00', 'A01', 'A02', 'A03', 'A04', 'A05', 'A06', 'A07', 'A08', 'A09', 'A10', 'A11', 'A12', 'A13', 'A14', 'A15', 'A16', 'A17', 'A18', 'A19', 'A20', 'A21', 'A22', 'A23', 'A24', 'A25', 'A26', 'A27', 'A28', 'A29', 'A30', 'A31', 'A32', 'A33', 'A34', 'A35', 'A36', 'A37', 'A38', 'A39', 'A40', 'A41', 'A42', 'A43', 'A44', 'A45', 'A46', 'A47', 'A48', 'A49', 'A50', 'A51', 'A52', 'A53', 'A54', 'A55', 'A56', 'A57', 'A58', 'A59', 'A60', 'A61', 'A62', 'A63']
Image Projection: {'type': 'Projection', 'crs': 'EPSG:32610', 'transform': [10, 0, 500000, 0, 10, 4259840]}

<b>2024 Image Details:</b>
Bands: ['A00', 'A01', 'A02', 'A03', 'A04', 'A05', 'A06', 'A07', 'A08', 'A09', 'A10', 'A11', 'A12', 'A13', 'A14', 'A15', 'A16', 'A17', 'A18', 'A19', 'A20', 'A21', 'A22', 'A23', 'A24', 'A25', 'A26', 'A27', 'A28', 'A29', 'A30', 'A31', 'A32', 'A33', 'A34', 'A35', 'A36', 'A37', 'A38', 'A39', 'A40', 'A41', 'A42', 'A43', 'A44', 'A45', 'A46', 'A47', 'A48', 'A49', 'A50', 'A51', 'A52', 'A

In [4]:
# 1. LOAD ALPHAEARTH EMBEDDINGS
alphaEarth = ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL')

# 2. DEFINE YOUR ANALYSIS PERIOD
startYear = 2018  # Match paper's June 2018 start
endYear = 2024    # Current available data

# 3. LOAD MINING LOCATIONS FROM CSV
# Data cite (Table S2? S3?): https://www.nature.com/articles/s41598-022-14987-0#MOESM3
# Sun, W., Jin, H., Jin, F. et al. Spatial analysis of global Bitcoin mining. Sci Rep 12, 10694 (2022). https://doi.org/10.1038/s41598-022-14987-0
# Data source: https://static-content.springer.com/esm/art%3A10.1038%2Fs41598-022-14987-0/MediaObjects/41598_2022_14987_MOESM3_ESM.xlsx
miningLocations = ee.FeatureCollection('projects/ee-ktwu01/assets/bitcoin-mining')


# Convert CSV data to proper format
def format_mining_location(feature):
    lat = ee.Number(feature.get('Latitude'))
    lon = ee.Number(feature.get('Longitude'))
    country = feature.get('CRCode')
    countryName = feature.get('CRName')

    return ee.Feature(
        ee.Geometry.Point([lon, lat]),
        {
            'label': 1,
            'year': 2018,
            'location': country,
            'country_name': countryName
        }
    )

miningLocations = miningLocations.map(format_mining_location)

# Print number of mining locations
numPositive = miningLocations.size()
print('Number of mining locations:', numPositive.getInfo())

# 4. VISUALIZE ALL POSITIVE MINING LOCATIONS ON MAP
# Create an interactive map
Map = geemap.Map(zoom=2)
Map.centerObject(miningLocations, 2)  # Global view
Map.addLayer(miningLocations, {'color': 'red', 'point_size': 0.0001}, 'Bitcoin Mining Locations')
Map

Number of mining locations: 6062


Map(center=[66.91775461601094, 26.123250690263546], controls=(WidgetControl(options=['position', 'transparent_…

In [5]:
# 5. GENERATE NEGATIVE SAMPLES (non-mining locations)
# Best practices for negative sampling:
# 1. Match the geographic distribution of positive samples
# 2. Include diverse land cover types (urban, rural, industrial, natural)
# 3. Use stratified random sampling within same regions
# 4. Aim for balanced dataset (1:1 or up to 1:3 positive:negative ratio)

# Get bounding box of mining locations to constrain negative sampling
miningBounds = miningLocations.geometry().bounds()

# Generate random points within the same geographic regions
negativeLocations = ee.FeatureCollection.randomPoints(
    region=miningBounds,
    points=numPositive,  # Match number of positive samples
    seed=42,  # For reproducibility
    maxError=1
)

# Add labels to negative samples
def add_negative_label(feature):
    return feature.set({
        'label': 0,
        'year': 2018,
        'location': 'negative_sample'
    })

negativeLocations = negativeLocations.map(add_negative_label)

# Optional: Filter out negative samples that are too close to mining sites
# This prevents contamination (e.g., excluding points within 1km of mining sites)
# minDistance = 1000  # meters

# The spatial filtering step is causing a persistent error ("String: Unable to convert object to string.").
# Commenting out this section as a workaround to allow the notebook to proceed.
# Note: Without this filtering, some negative samples may be very close to mining locations.
#
# # Create buffers around each mining location and merge them
# miningBuffers = miningLocations.map(lambda f: f.buffer(minDistance)).flatten()
# mergedMiningBuffer = miningBuffers.union()
#
# # Filter negative samples to exclude those that intersect with the merged buffer
# negativeLocations = negativeLocations.filter(ee.Filter.disjoint(mergedMiningBuffer))


print('Number of negative samples after filtering:', negativeLocations.size().getInfo())

# 6. COMBINE POSITIVE AND NEGATIVE SAMPLES
trainingPoints = miningLocations.merge(negativeLocations)
print('Total training points:', trainingPoints.size().getInfo())

Number of negative samples after filtering: 6062
Total training points: 12124


In [6]:
# # 7. EXTRACT EMBEDDINGS AT TRAINING POINTS FOR SPECIFIC YEAR
# # running four minutes and half after Extracting embeddings for multiple years...
# # # Extracting embeddings for multiple years...
# # Number of images for year 2018: 0
# # Number of images for year 2021: 0
# # Number of images for year 2024: 0
# # Embeddings extracted successfully!
# # This cannot be true. There are no numbers for these years.
# # We should debug for this.

# def extractEmbeddings(year):
#     """Extract AlphaEarth embeddings for a specific year by date range"""
#     print(f"--- Debugging extractEmbeddings for year {year} ---")
#     start_date = f"{year}-01-01"
#     end_date = f"{year + 1}-01-01"
#     print(f"Filtering AlphaEarth collection for date range: {start_date} to {end_date}")
#     yearImageCollection = alphaEarth.filterDate(start_date, end_date)
#     print(f"Number of images for year {year} in date range: {yearImageCollection.size().getInfo()}")
#     yearImage = yearImageCollection.first()
#     print(f"First image found for year {year}: {yearImage.getInfo() if yearImage else 'None'}")

#     samples = ee.FeatureCollection([]) # Initialize empty collection
#     if yearImage:
#         print(f"Filtering training points by image bounds for year {year}...")
#         # Filter training points to only include those within the bounds of the image
#         filteredTrainingPoints = trainingPoints.filterBounds(yearImage.geometry())
#         print(f"Number of training points within image bounds for year {year}: {filteredTrainingPoints.size().getInfo()}")

#         if filteredTrainingPoints.size().getInfo() > 0:
#             print(f"Sampling regions for year {year}...")
#             samples = yearImage.sampleRegions(
#                 collection=filteredTrainingPoints,
#                 scale=10,  # AlphaEarth is 10m resolution
#                 geometries=True
#             )
#             print(f"Number of samples extracted for year {year}: {samples.size().getInfo()}")
#             print(f"Sample features for year {year} (first 5): {samples.limit(5).getInfo()}")
#         else:
#             print(f"No training points within image bounds for year {year}, returning empty samples.")
#     else:
#         print(f"No image found for year {year} in date range, returning empty samples.")

#     print(f"--- Finished extractEmbeddings for year {year} ---")
#     return samples

# # 8. EXTRACT FOR MULTIPLE YEARS
# print('--- Debugging Extraction for Multiple Years ---')
# print('Extracting embeddings for multiple years...')
# embeddings2018 = extractEmbeddings(2018)
# embeddings2021 = extractEmbeddings(2021)  # Pre-China ban
# embeddings2024 = extractEmbeddings(2024)  # Post-ban

# print('Embeddings extracted successfully!')
# print('2018 samples:', embeddings2018.size().getInfo())
# print('2021 samples:', embeddings2021.size().getInfo())
# print('2024 samples:', embeddings2024.size().getInfo())
# print('--- Finished Extraction for Multiple Years ---')

# # 9. EXPORT FOR CLASSIFIER TRAINING
# print('--- Debugging Export Configuration ---')
# # Generate band selectors for all 64 AlphaEarth layers
# band_names = ['A' + str(i).zfill(2) for i in range(64)]
# selectors = ['label', 'location', 'year'] + band_names
# print(f"Selectors for export: {selectors}")

# # Export 2018 data to Google Drive
# task2018 = ee.batch.Export.table.toDrive(
#     collection=embeddings2018,
#     description='AlphaEarth_Mining_Training_2018',
#     fileFormat='CSV',
#     selectors=selectors
# )

# # Uncomment to start the export task
# # task2018.start()
# # print('Export task started. Check your Google Drive and Earth Engine Tasks tab.')

# print('Export task configured. To start export, uncomment task2018.start() and run again.')
# print(f'This will export {embeddings2018.size().getInfo()} samples with {len(band_names)} embedding bands.')
# print('--- Finished Export Configuration ---')

# Task
Review the provided Earth Engine algorithm for extracting embeddings, identify the part that is running correctly (defining selectors), and create a new single cell with a revised approach for embedding extraction using a mosaic method, including the necessary debug output.

## Address timeout in embedding extraction

### Subtask:
Address the timeout issue encountered during the extraction of AlphaEarth embeddings using the mosaic approach. The previous attempt to sample all training points within the mosaic's bounds timed out. This subtask focuses on implementing a more efficient method to extract embeddings without hitting the computation limit.


**Reasoning**:
The previous attempt to get the size of the sampled FeatureCollection timed out. The core issue is likely sampling a large number of points over a large mosaic. The export function can handle large computations in the background. Therefore, modify the extraction function to return the sampled FeatureCollection directly without attempting to compute its size or get its info within the function. This will allow the export task to handle the computation in the background.



In [7]:
# # 7. EXTRACT EMBEDDINGS AT TRAINING POINTS FOR SPECIFIC YEAR (Revised Mosaic approach)

# def extractEmbeddings_mosaic_revised(year):
#     """Extract AlphaEarth embeddings for a specific year using a mosaic,
#        returning the FeatureCollection without immediate computation."""
#     print(f"--- Starting extractEmbeddings_mosaic_revised for year {year} ---")
#     start_date = f"{year}-01-01"
#     end_date = f"{year + 1}-01-01"
#     print(f"Filtering AlphaEarth collection for date range: {start_date} to {end_date}")

#     # Filter the collection by date
#     yearImageCollection = alphaEarth.filterDate(start_date, end_date)
#     print(f"Number of images for year {year} in date range: {yearImageCollection.size().getInfo()}")

#     samples = ee.FeatureCollection([]) # Initialize empty collection

#     if yearImageCollection.size().getInfo() > 0:
#         # Create a mosaic of the images for the year
#         yearImage = yearImageCollection.mosaic()
#         print(f"Mosaic image created for year {year}.")

#         print(f"Filtering training points by mosaic bounds for year {year}...")
#         # Filter training points to only include those within the bounds of the mosaic
#         filteredTrainingPoints = trainingPoints.filterBounds(yearImage.geometry())
#         num_filtered_points = filteredTrainingPoints.size().getInfo() # Still get this size for logging/debugging
#         print(f"Number of training points within mosaic bounds for year {year}: {num_filtered_points}")


#         if num_filtered_points > 0:
#             print(f"Configuring sampleRegions for year {year}...")
#             # Perform the sampling. This returns a FeatureCollection, but we avoid
#             # calling getInfo() on it here.
#             samples = yearImage.sampleRegions(
#                 collection=filteredTrainingPoints,
#                 scale=10,  # AlphaEarth is 10m resolution
#                 geometries=True
#             )
#             print(f"SampleRegions configured for year {year}. Returning FeatureCollection.")
#         else:
#             print(f"No training points within mosaic bounds for year {year}, returning empty samples collection.")
#     else:
#         print(f"No images found for year {year} in date range, returning empty samples collection.")

#     print(f"--- Finished extractEmbeddings_mosaic_revised for year {year} ---")
#     return samples

# # 8. EXTRACT FOR MULTIPLE YEARS (using revised mosaic approach)
# print('--- Debugging Extraction for Multiple Years (Revised Mosaic Approach) ---')
# print('Extracting embeddings for multiple years...')
# embeddings2018_mosaic = extractEmbeddings_mosaic_revised(2018)
# embeddings2021_mosaic = extractEmbeddings_mosaic_revised(2021)  # Pre-China ban
# embeddings2024_mosaic = extractEmbeddings_mosaic_revised(2024)  # Post-ban

# print('Embeddings extraction configured successfully! Ready for export.')
# # We don't print the size here to avoid the timeout
# # print('2018 samples (mosaic):', embeddings2018_mosaic.size().getInfo())
# # print('2021 samples (mosaic):', embeddings2021_mosaic.size().getInfo())
# # print('2024 samples (mosaic):', embeddings2024_mosaic.size().getInfo())
# print('--- Finished Extraction for Multiple Years (Revised Mosaic Approach) ---')

**Reasoning**:
The embedding extraction functions have been revised to return the sampled FeatureCollections without triggering immediate computation, addressing the timeout issue. The next logical step is to configure the export tasks using these returned FeatureCollections. The previous attempt to configure export failed because the variables holding the FeatureCollections were not defined due to the timeout. This step will re-run the export configuration code, which should now work since the necessary FeatureCollections are available.



In [8]:
# # 9. EXPORT FOR CLASSIFIER TRAINING (using revised mosaic approach)
# print('--- Debugging Export Configuration (Revised Mosaic Approach) ---')
# # Generate band selectors for all 64 AlphaEarth layers
# band_names = ['A' + str(i).zfill(2) for i in range(64)]
# selectors = ['label', 'location', 'year'] + band_names
# print(f"Selectors for export: {selectors}")

# # Export 2018 data to Google Drive
# task2018_mosaic = ee.batch.Export.table.toDrive(
#     collection=embeddings2018_mosaic,
#     description='AlphaEarth_Mining_Training_2018_Mosaic_Revised',
#     folder='AlphaEarth_Mining_Training', # Optional: specify a folder in Drive
#     fileFormat='CSV',
#     selectors=selectors
# )

# # Export 2021 data to Google Drive
# task2021_mosaic = ee.batch.Export.table.toDrive(
#     collection=embeddings2021_mosaic,
#     description='AlphaEarth_Mining_Training_2021_Mosaic_Revised',
#     folder='AlphaEarth_Mining_Training', # Optional: specify a folder in Drive
#     fileFormat='CSV',
#     selectors=selectors
# )

# # Export 2024 data to Google Drive
# task2024_mosaic = ee.batch.Export.table.toDrive(
#     collection=embeddings2024_mosaic,
#     description='AlphaEarth_Mining_Training_2024_Mosaic_Revised',
#     folder='AlphaEarth_Mining_Training', # Optional: specify a folder in Drive
#     fileFormat='CSV',
#     selectors=selectors
# )

# # Uncomment to start the export tasks
# # task2018_mosaic.start()
# # task2021_mosaic.start()
# # task2024_mosaic.start()
# # print('Export tasks started. Check your Google Drive and Earth Engine Tasks tab.')

# print('Export tasks configured. To start export, uncomment the task.start() lines and run again.')
# print(f'This will configure export for 2018, 2021, and 2024 samples with {len(band_names)} embedding bands.')
# print('--- Finished Export Configuration (Revised Mosaic Approach) ---')

## Create cell for successful code

### Subtask:
Move the code that defines the selectors (which ran successfully) into its own cell.


**Reasoning**:
Move the code that defines the band names and selectors into a new cell as requested.



In [9]:
# Define band names and selectors for export

# Generate band selectors for all 64 AlphaEarth layers
band_names = ['A' + str(i).zfill(2) for i in range(64)]
selectors = ['label', 'location', 'year'] + band_names
print(f"Selectors for export: {selectors}")

Selectors for export: ['label', 'location', 'year', 'A00', 'A01', 'A02', 'A03', 'A04', 'A05', 'A06', 'A07', 'A08', 'A09', 'A10', 'A11', 'A12', 'A13', 'A14', 'A15', 'A16', 'A17', 'A18', 'A19', 'A20', 'A21', 'A22', 'A23', 'A24', 'A25', 'A26', 'A27', 'A28', 'A29', 'A30', 'A31', 'A32', 'A33', 'A34', 'A35', 'A36', 'A37', 'A38', 'A39', 'A40', 'A41', 'A42', 'A43', 'A44', 'A45', 'A46', 'A47', 'A48', 'A49', 'A50', 'A51', 'A52', 'A53', 'A54', 'A55', 'A56', 'A57', 'A58', 'A59', 'A60', 'A61', 'A62', 'A63']


## Create cell for embedding extraction (revised)

### Subtask:
Create a new cell for the embedding extraction logic, incorporating the changes to address the timeout issue.


**Reasoning**:
Define the function for extracting embeddings using the mosaic approach and call it for the specified years.



In [10]:
# # 7. EXTRACT EMBEDDINGS AT TRAINING POINTS FOR SPECIFIC YEAR (Revised Mosaic approach)

# def extractEmbeddings_mosaic_revised(year):
#     """Extract AlphaEarth embeddings for a specific year using a mosaic,
#        returning the FeatureCollection without immediate computation."""
#     print(f"--- Starting extractEmbeddings_mosaic_revised for year {year} ---")
#     start_date = f"{year}-01-01"
#     end_date = f"{year + 1}-01-01"
#     print(f"Filtering AlphaEarth collection for date range: {start_date} to {end_date}")

#     # Filter the collection by date
#     yearImageCollection = alphaEarth.filterDate(start_date, end_date)
#     collection_size = yearImageCollection.size().getInfo()
#     print(f"Number of images for year {year} in date range: {collection_size}")

#     samples = ee.FeatureCollection([]) # Initialize empty collection

#     if collection_size > 0:
#         # Create a mosaic of the images for the year
#         yearImage = yearImageCollection.mosaic()
#         print(f"Mosaic image created for year {year}.")

#         print(f"Filtering training points by mosaic bounds for year {year}...")
#         # Filter training points to only include those within the bounds of the mosaic
#         filteredTrainingPoints = trainingPoints.filterBounds(yearImage.geometry())
#         num_filtered_points = filteredTrainingPoints.size().getInfo() # Still get this size for logging/debugging
#         print(f"Number of training points within mosaic bounds for year {year}: {num_filtered_points}")


#         if num_filtered_points > 0:
#             print(f"Configuring sampleRegions for year {year}...")
#             # Perform the sampling. This returns a FeatureCollection, but we avoid
#             # calling getInfo() on it here.
#             samples = yearImage.sampleRegions(
#                 collection=filteredTrainingPoints,
#                 scale=10,  # AlphaEarth is 10m resolution
#                 geometries=True
#             )
#             print(f"SampleRegions configured for year {year}. Returning FeatureCollection.")
#         else:
#             print(f"No training points within mosaic bounds for year {year}, returning empty samples collection.")
#     else:
#         print(f"No images found for year {year} in date range, returning empty samples collection.")

#     print(f"--- Finished extractEmbeddings_mosaic_revised for year {year} ---")
#     return samples

# # 8. EXTRACT FOR MULTIPLE YEARS (using revised mosaic approach)
# print('--- Debugging Extraction for Multiple Years (Revised Mosaic Approach) ---')
# print('Extracting embeddings for multiple years...')
# embeddings2018_mosaic = extractEmbeddings_mosaic_revised(2018)
# embeddings2021_mosaic = extractEmbeddings_mosaic_revised(2021)  # Pre-China ban
# embeddings2024_mosaic = extractEmbeddings_mosaic_revised(2024)  # Post-ban

# print('Embeddings extraction configured successfully! Ready for export.')
# print('--- Finished Extraction for Multiple Years (Revised Mosaic Approach) ---')

## Create cell for export configuration

### Subtask:
Create a cell for the export configuration using the sampled embeddings from the revised extraction method.


**Reasoning**:
Configure the export tasks for the sampled embeddings using the previously defined selectors.



In [11]:
# # 9. EXPORT FOR CLASSIFIER TRAINING (using mosaic approach)
# print('--- Debugging Export Configuration (Mosaic Approach) ---')

# # Export 2018 data to Google Drive
# task2018_mosaic = ee.batch.Export.table.toDrive(
#     collection=embeddings2018_mosaic,
#     description='AlphaEarth_Mining_Training_2018_Mosaic_Revised',
#     folder='AlphaEarth_Mining_Training', # Optional: specify a folder in Drive
#     fileFormat='CSV',
#     selectors=selectors
# )

# # Export 2021 data to Google Drive
# task2021_mosaic = ee.batch.Export.table.toDrive(
#     collection=embeddings2021_mosaic,
#     description='AlphaEarth_Mining_Training_2021_Mosaic_Revised',
#     folder='AlphaEarth_Mining_Training', # Optional: specify a folder in Drive
#     fileFormat='CSV',
#     selectors=selectors
# )

# # Export 2024 data to Google Drive
# task2024_mosaic = ee.batch.Export.table.toDrive(
#     collection=embeddings2024_mosaic,
#     description='AlphaEarth_Mining_Training_2024_Mosaic_Revised',
#     folder='AlphaEarth_Mining_Training', # Optional: specify a folder in Drive
#     fileFormat='CSV',
#     selectors=selectors
# )

# # Uncomment to start the export tasks
# # task2018_mosaic.start()
# # task2021_mosaic.start()
# # task2024_mosaic.start()
# # print('Export tasks started. Check your Google Drive and Earth Engine Tasks tab.')

# print('Export tasks configured. To start export, uncomment the task.start() lines and run again.')
# print(f'This will configure export for 2018, 2021, and 2024 samples with {len(band_names)} embedding bands.')
# print('--- Finished Export Configuration (Mosaic Approach) ---')

## Update debugging output

### Subtask:
Update debugging output to reflect the revised embedding extraction process and the successful configuration of export tasks.


**Reasoning**:
Review the existing print statements related to embedding extraction and export configuration and update them to accurately reflect the revised mosaic approach and the deferred computation.



In [12]:
# 7. EXTRACT EMBEDDINGS AT TRAINING POINTS FOR SPECIFIC YEAR (Revised Mosaic approach)

def extractEmbeddings_mosaic_revised(year):
    """Extract AlphaEarth embeddings for a specific year using a mosaic,
       returning the FeatureCollection without immediate computation."""
    print(f"--- Starting embedding extraction configuration for year {year} (Revised Mosaic approach) ---")
    start_date = f"{year}-01-01"
    end_date = f"{year + 1}-01-01"
    print(f"Filtering AlphaEarth collection for date range: {start_date} to {end_date}")

    # Filter the collection by date
    yearImageCollection = alphaEarth.filterDate(start_date, end_date)
    collection_size = yearImageCollection.size().getInfo()
    print(f"Number of images found for year {year} in date range: {collection_size}")

    samples = ee.FeatureCollection([]) # Initialize empty collection

    if collection_size > 0:
        # Create a mosaic of the images for the year
        yearImage = yearImageCollection.mosaic()
        print(f"Mosaic image created for year {year}.")

        print(f"Filtering training points by mosaic bounds for year {year}...")
        # Filter training points to only include those within the bounds of the mosaic
        filteredTrainingPoints = trainingPoints.filterBounds(yearImage.geometry())
        # Getting the size here is acceptable as it's on the training points, not the sampled image data
        num_filtered_points = filteredTrainingPoints.size().getInfo()
        print(f"Number of training points within mosaic bounds for year {year}: {num_filtered_points}")


        if num_filtered_points > 0:
            print(f"Configuring sampleRegions operation for year {year}...")
            # Perform the sampling. This returns a FeatureCollection, but we avoid
            # calling getInfo() on it here, deferring computation.
            samples = yearImage.sampleRegions(
                collection=filteredTrainingPoints,
                scale=10,  # AlphaEarth is 10m resolution
                geometries=True
            )
            print(f"SampleRegions configured for year {year}. Returning FeatureCollection for later export.")
        else:
            print(f"No training points within mosaic bounds for year {year}, returning empty samples collection.")
    else:
        print(f"No images found for year {year} in date range, returning empty samples collection.")

    print(f"--- Finished embedding extraction configuration for year {year} ---")
    return samples

# 8. EXTRACT FOR MULTIPLE YEARS (using revised mosaic approach)
print('--- Configuring Embedding Extraction for Multiple Years (Revised Mosaic Approach) ---')
print('Configuring embedding extraction tasks...')
embeddings2018_mosaic = extractEmbeddings_mosaic_revised(2018)
embeddings2021_mosaic = extractEmbeddings_mosaic_revised(2021)  # Pre-China ban
embeddings2024_mosaic = extractEmbeddings_mosaic_revised(2024)  # Post-ban

print('Embedding extraction configurations created successfully! These are FeatureCollections ready for export.')
print('--- Finished Configuring Embedding Extraction for Multiple Years ---')

# Define band names and selectors for export (already in a separate cell)
# Generate band selectors for all 64 AlphaEarth layers
# band_names = ['A' + str(i).zfill(2) for i in range(64)]
# selectors = ['label', 'location', 'year'] + band_names
# print(f"Selectors for export: {selectors}")


# 9. EXPORT FOR CLASSIFIER TRAINING (using mosaic approach)
print('\n--- Configuring Export Tasks (Revised Mosaic Approach) ---')

# Export 2018 data to Google Drive
task2018_mosaic = ee.batch.Export.table.toDrive(
    collection=embeddings2018_mosaic,
    description='AlphaEarth_Mining_Training_2018_Mosaic_Revised',
    folder='AlphaEarth_Mining_Training', # Optional: specify a folder in Drive
    fileFormat='CSV',
    selectors=selectors
)
print('Export task configured for 2018 data.')

# Export 2021 data to Google Drive
task2021_mosaic = ee.batch.Export.table.toDrive(
    collection=embeddings2021_mosaic,
    description='AlphaEarth_Mining_Training_2021_Mosaic_Revised',
    folder='AlphaEarth_Mining_Training', # Optional: specify a folder in Drive
    fileFormat='CSV',
    selectors=selectors
)
print('Export task configured for 2021 data.')


# Export 2024 data to Google Drive
task2024_mosaic = ee.batch.Export.table.toDrive(
    collection=embeddings2024_mosaic,
    description='AlphaEarth_Mining_Training_2024_Mosaic_Revised',
    folder='AlphaEarth_Mining_Training', # Optional: specify a folder in Drive
    fileFormat='CSV',
    selectors=selectors
)
print('Export task configured for 2024 data.')


--- Configuring Embedding Extraction for Multiple Years (Revised Mosaic Approach) ---
Configuring embedding extraction tasks...
--- Starting embedding extraction configuration for year 2018 (Revised Mosaic approach) ---
Filtering AlphaEarth collection for date range: 2018-01-01 to 2019-01-01
Number of images found for year 2018 in date range: 10635
Mosaic image created for year 2018.
Filtering training points by mosaic bounds for year 2018...
Number of training points within mosaic bounds for year 2018: 12124
Configuring sampleRegions operation for year 2018...
SampleRegions configured for year 2018. Returning FeatureCollection for later export.
--- Finished embedding extraction configuration for year 2018 ---
--- Starting embedding extraction configuration for year 2021 (Revised Mosaic approach) ---
Filtering AlphaEarth collection for date range: 2021-01-01 to 2022-01-01
Number of images found for year 2021 in date range: 10614
Mosaic image created for year 2021.
Filtering training po

In [13]:
print('\nExport tasks configured successfully. To start export, uncomment the task.start() lines and run again.')
print(f'This will export the configured 2018, 2021, and 2024 samples with {len(band_names)} embedding bands when the tasks are started.')
print('--- Finished Configuring Export Tasks (Revised Mosaic Approach) ---')

# Uncomment to start the export tasks
task2018_mosaic.start()
task2021_mosaic.start()
task2024_mosaic.start()
print('Export tasks started. Check your Google Drive and Earth Engine Tasks tab.')

# # monitor the progress of the export tasks in the Earth Engine Tasks tab
# Submitted tasks
# AlphaEarth_Mining_Training_2024_Mosaic_Revised
# ID: MH5WWJSOVGEH5V5OMBMA54RC
# Phase: Completed
# Runtime: 2m (started 2025-10-11 17:22:47 -0500)
# Attempted 1 time
# Priority: 100 (default)
# Batch compute usage: 106724.1875 EECU-seconds
# 2m
# AlphaEarth_Mining_Training_2021_Mosaic_Revised
# ID: UV7HHKGEVSVR4XORYODI4CJL
# Phase: Completed
# Runtime: 3m (started 2025-10-11 17:22:36 -0500)
# Attempted 1 time
# Priority: 100 (default)
# Batch compute usage: 90087.0547 EECU-seconds
# 3m
# AlphaEarth_Mining_Training_2018_Mosaic_Revised
# ID: I6YEDMAHD223QEBU6MNOKG72
# Phase: Completed
# Runtime: 2m (started 2025-10-11 17:22:36 -0500)
# Attempted 1 time
# Priority: 100 (default)
# Batch compute usage: 106359.8047 EECU-seconds



Export tasks configured successfully. To start export, uncomment the task.start() lines and run again.
This will export the configured 2018, 2021, and 2024 samples with 64 embedding bands when the tasks are started.
--- Finished Configuring Export Tasks (Revised Mosaic Approach) ---
Export tasks started. Check your Google Drive and Earth Engine Tasks tab.


## Summary:

### Data Analysis Key Findings

*   The part of the algorithm that defines the selectors for the 64 AlphaEarth embedding bands ('A00' to 'A63') along with 'label', 'location', and 'year' was correctly identified and is functional.
*   The revised mosaic approach for embedding extraction successfully configures the `sampleRegions` operation and returns an Earth Engine `FeatureCollection` without triggering immediate computation, thus avoiding previous timeout issues.
*   Debugging output confirms the number of images found for each year and the number of training points within the mosaic bounds.
*   The export tasks for the sampled embeddings from 2018, 2021, and 2024 are successfully configured using the generated selectors and the returned `FeatureCollection` objects.
*   The revised process defers the heavy computation of sampling and exporting the data to the Earth Engine background tasks, which are initiated when `task.start()` is called.

### Insights or Next Steps

*   The revised method effectively overcomes the computation limit by deferring the sampling and export computations to Earth Engine's background processing.
*   The next crucial step is to uncomment the `task.start()` lines in the export configuration cell to initiate the export of the sampled AlphaEarth embeddings to Google Drive for subsequent use in classifier training.


In [14]:

# 10. VISUALIZE TRAINING POINTS ON MAP
print('--- Debugging Visualization ---')
# Create a comprehensive visualization
Map2 = geemap.Map(zoom=2)
print("Geemap map created.")

# Get AlphaEarth image for 2018 by creating a median composite over a date range
# Ensure an image is obtained, even if no image has a 'year' property set to 2018
ae2018_collection = alphaEarth.filterDate('2018-01-01', '2019-01-01')
print(f"Number of images in 2018 date range collection: {ae2018_collection.size().getInfo()}")
ae2018 = ae2018_collection.median() if ae2018_collection.size().getInfo() > 0 else None
print(f"Median composite image for 2018: {ae2018.getInfo() if ae2018 else 'None'}")


# Add AlphaEarth as RGB (using bands A01, A16, A09)
vis_params = {
    'min': -0.3,
    'max': 0.3,
    'bands': ['A01', 'A16', 'A09']
}
print(f"Visualization parameters: {vis_params}")

if ae2018:
    Map2.addLayer(ae2018, vis_params, 'AlphaEarth 2018 RGB', False)
    print("AlphaEarth 2018 RGB layer added.")
else:
    print("No AlphaEarth 2018 image available to add as a layer.")

Map2.addLayer(miningLocations, {'color': 'FF0000'}, 'Mining Locations (Positive)')
print("Mining Locations layer added.")
Map2.addLayer(negativeLocations, {'color': '0000FF'}, 'Negative Samples')
print("Negative Samples layer added.")

# Center on first mining location
if miningLocations.size().getInfo() > 0:
    Map2.centerObject(miningLocations.first(), 12)
    print("Map centered on first mining location.")
else:
    print("No mining locations to center the map on.")


print('Training Points Summary:')
print('  Total:', trainingPoints.size().getInfo())
print('  Positive (Mining):', miningLocations.size().getInfo())
print('  Negative (Non-mining):', negativeLocations.size().getInfo())
if ae2018:
    print('\nAlphaEarth Bands:', ae2018.bandNames().getInfo())
else:
    print('\nAlphaEarth image not available, band names cannot be retrieved.')
print('\nMap Legend:')
print('  Red points = Bitcoin mining locations')
print('  Blue points = Negative samples (non-mining)')

print('--- Finished Visualization ---')
Map2

--- Debugging Visualization ---
Geemap map created.
Number of images in 2018 date range collection: 10635
Median composite image for 2018: {'type': 'Image', 'bands': [{'id': 'A00', 'data_type': {'type': 'PixelType', 'precision': 'double'}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'A01', 'data_type': {'type': 'PixelType', 'precision': 'double'}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'A02', 'data_type': {'type': 'PixelType', 'precision': 'double'}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'A03', 'data_type': {'type': 'PixelType', 'precision': 'double'}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'A04', 'data_type': {'type': 'PixelType', 'precision': 'double'}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'A05', 'data_type': {'type': 'PixelType', 'precision': 'double'}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'A06', 'data_type': {'type': 'PixelType'

Map(center=[34.34817123, 62.19966888], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=…

In [15]:
from google.colab import drive
drive.mount('/content/drive/',force_remount=True)  # Change folder name here


Mounted at /content/drive/


In [16]:
%%bash
cd /content/drive/My\ Drive/AlphaEarthHack
ls -la data/training/

total 32384
-rw------- 1 root root 11050210 Oct 11 22:24 AlphaEarth_Mining_Training_2018_Mosaic_Revised.csv
-rw------- 1 root root 11054620 Oct 11 22:25 AlphaEarth_Mining_Training_2021_Mosaic_Revised.csv
-rw------- 1 root root 11054698 Oct 11 22:24 AlphaEarth_Mining_Training_2024_Mosaic_Revised.csv


In [17]:
import pandas as pd
import os

# Define the path to your Google Drive folder
drive_path = '/content/drive/My Drive/AlphaEarthHack/data/training/' # Assuming the data is in this subfolder

# Load the CSV files into pandas DataFrames
df_2018 = pd.read_csv(os.path.join(drive_path, 'AlphaEarth_Mining_Training_2018_Mosaic_Revised.csv'))
df_2021 = pd.read_csv(os.path.join(drive_path, 'AlphaEarth_Mining_Training_2021_Mosaic_Revised.csv'))
df_2024 = pd.read_csv(os.path.join(drive_path, 'AlphaEarth_Mining_Training_2024_Mosaic_Revised.csv'))

# Add a 'point_id' column to each dataframe based on the original index
df_2018['point_id'] = df_2018.index
df_2021['point_id'] = df_2021.index
df_2024['point_id'] = df_2024.index


print("DataFrames loaded successfully:")
print("df_2018 shape:", df_2018.shape)
print("df_2021 shape:", df_2021.shape)
print("df_2024 shape:", df_2024.shape)

# Display the first few rows of each DataFrame to inspect the data
print("\nFirst 5 rows of df_2018:")
display(df_2018.head())

print("\nFirst 5 rows of df_2021:")
display(df_2021.head())

print("\nFirst 5 rows of df_2024:")
display(df_2024.head())

DataFrames loaded successfully:
df_2018 shape: (8324, 68)
df_2021 shape: (8328, 68)
df_2024 shape: (8328, 68)

First 5 rows of df_2018:


,label,location,year,A00,A01,A02,A03,A04,A05,A06,...,A55,A56,A57,A58,A59,A60,A61,A62,A63,point_id
0,1,NaN,2018,0.186082,-0.038447,0.192910,-0.044844,0.017778,0.022207,0.135886,...,-0.013841,-0.035433,0.010396,-0.048228,-0.024606,0.055363,-0.135886,-0.153787,0.027128,0
1,1,NaN,2018,0.103406,-0.228897,0.055363,0.027128,0.192910,0.012057,-0.022207,...,-0.108512,-0.098424,-0.103406,-0.236463,-0.153787,0.179377,-0.022207,-0.141730,0.029773,1
2,1,NaN,2018,0.221453,-0.244152,0.032541,-0.088827,0.059116,-0.013841,0.228897,...,-0.130165,0.044844,0.062991,-0.055363,0.066990,0.228897,-0.041584,-0.327812,0.124567,2
3,1,NaN,2018,0.041584,-0.166336,0.186082,-0.059116,0.135886,-0.017778,-0.017778,...,-0.236463,0.055363,-0.079723,0.135886,0.008858,0.103406,-0.019931,-0.006151,-0.113741,3
4,1,NaN,2018,-0.027128,-0.228897,0.251965,-0.098424,0.013841,-0.062991,0.035433,...,-0.199862,0.006151,-0.066990,0.015748,-0.055363,0.079723,-0.179377,-0.038447,-0.130165,4



First 5 rows of df_2021:


,label,location,year,A00,A01,A02,A03,A04,A05,A06,...,A55,A56,A57,A58,A59,A60,A61,A62,A63,point_id
0,1,NaN,2018,0.199862,-0.051734,0.186082,-0.066990,0.024606,0.051734,0.147697,...,-0.038447,-0.038447,-0.010396,-0.029773,-0.032541,0.024606,-0.119093,-0.153787,0.022207,0
1,1,NaN,2018,0.103406,-0.267958,0.079723,0.048228,0.153787,-0.017778,-0.044844,...,-0.119093,-0.098424,-0.088827,-0.236463,-0.130165,0.199862,-0.022207,-0.141730,0.048228,1
2,1,NaN,2018,0.236463,-0.206936,0.019931,-0.075356,0.044844,-0.013841,0.236463,...,-0.113741,0.044844,0.044844,-0.048228,0.071111,0.214133,-0.022207,-0.355309,0.141730,2
3,1,NaN,2018,0.048228,-0.166336,0.166336,-0.024606,0.108512,-0.041584,-0.019931,...,-0.236463,0.066990,-0.066990,0.119093,-0.032541,0.119093,-0.012057,-0.024606,-0.119093,3
4,1,NaN,2018,-0.001538,-0.284444,0.236463,-0.022207,0.032541,-0.079723,0.051734,...,-0.236463,0.010396,-0.098424,0.041584,0.059116,0.153787,-0.084214,-0.119093,-0.108512,4



First 5 rows of df_2024:


,label,location,year,A00,A01,A02,A03,A04,A05,A06,...,A55,A56,A57,A58,A59,A60,A61,A62,A63,point_id
0,1,NaN,2018,0.186082,-0.062991,0.192910,-0.051734,0.098424,0.035433,0.160000,...,-0.084214,0.006151,-0.027128,-0.038447,-0.015748,0.062991,-0.160000,-0.119093,0.019931,0
1,1,NaN,2018,0.088827,-0.327812,0.119093,0.010396,0.206936,0.012057,-0.041584,...,-0.166336,-0.066990,-0.103406,-0.228897,-0.075356,0.186082,-0.019931,-0.130165,0.048228,1
2,1,NaN,2018,0.244152,-0.236463,0.019931,-0.044844,0.059116,-0.010396,0.206936,...,-0.130165,0.062991,0.071111,-0.075356,0.059116,0.236463,-0.024606,-0.355309,0.141730,2
3,1,NaN,2018,0.066990,-0.179377,0.206936,-0.051734,0.084214,0.001538,0.000554,...,-0.236463,0.088827,-0.071111,0.135886,0.003014,0.075356,0.012057,0.002215,-0.113741,3
4,1,NaN,2018,0.019931,-0.276140,0.318893,-0.022207,0.059116,-0.035433,0.066990,...,-0.228897,0.041584,-0.075356,0.010396,0.055363,0.113741,-0.066990,-0.079723,-0.079723,4


1. **Analyze Embedding Changes at Known Sites:** Continue the analysis of the calculated embedding changes (`df_embedding_changes`) by comparing metrics for mining vs non-mining sites globally.
2. **Investigate Changes in Specific Regions:** Filter the `df_embedding_changes` DataFrame to focus on regions of interest (e.g., China, US, Kazakhstan) and analyze the patterns of embedding changes within these regions.
3. **Interpret Findings:** Synthesize the results from the regional analysis to gain insights into how the characteristics of known mining sites have evolved, potentially correlating these changes with known events like the China ban.
4. **Visualize Embedding Changes:** Create visualizations (e.g., histograms, box plots, scatter plots) to illustrate the distribution of embedding change metrics for different groups (mining vs. non-mining, different regions).
5. **Finish task**: Summarize the findings and the methodology.

## Current Status and Plan

We have successfully:
1. Loaded the Bitcoin mining location data and generated negative samples.
2. Extracted AlphaEarth embeddings for 2018, 2021, and 2024 using the mosaic approach and configured the export tasks. The exported data is now in your Google Drive.
3. Loaded the exported embedding data into pandas DataFrames.
4. Prepared the data for training by combining the data from different years, separating features (embeddings) and labels (mining vs. non-mining), and splitting the data into training and testing sets.
5. Trained a RandomForestClassifier model on the combined 2018, 2021, and 2024 data.

**Understanding the Model and Limitations:**

It's important to remember that the current training data uses static labels based on 2018 mining locations. The trained model will identify locations with characteristics similar to 2018 mining sites based on AlphaEarth embeddings. It does *not* directly tell us how mining activity has changed over time or whether a location is *currently* an active mining site in 2021 or 2024.

**Revised Plan for Analyzing Change Over Time:**

To address the goal of understanding how Bitcoin mining locations change over time, given the static nature of our current label data, we will shift our focus to analyzing the *changes in AlphaEarth embeddings* at the known 2018 sites. The revised plan is as follows:

1. Useless: we only have one year data. not do this. Evaluate the Classifier (on Test Data): Evaluate the performance of the currently trained model on the unseen test data (`X_test`, `y_test`) to understand its ability to generalize.
2.  **Analyze Embedding Changes at Known Sites:** Extract the AlphaEarth embeddings for the known 2018 mining and non-mining locations for multiple years (2018, 2021, 2024 - which we have already extracted). Then, analyze the patterns of change in these embeddings over time at these specific locations. This could involve:
    *   Calculating metrics of change (e.g., difference, cosine similarity) between embeddings from different years.
    *   Visualizing the embedding changes.
    *   Potentially using clustering or other techniques to identify groups of sites with similar change patterns.
3.  **Investigate Changes in Specific Regions:** Focus the analysis on regions of interest (e.g., China pre- and post-ban) to see if the embedding changes at known mining sites correlate with expected changes in activity.
4.  **Interpret Findings:** Synthesize the results to gain insights into how the characteristics of known mining sites have evolved based on AlphaEarth embeddings, providing indirect evidence of changes in activity.
5.  **Finish task**: Summarize the findings and the methodology.

In [20]:
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd # Ensure pandas is imported

# Combine the data from all three years
# Use merge instead of concat to align data by point_id
# Use suffixes to distinguish columns from different years
df_merged_2018_2021 = pd.merge(df_2018, df_2021, on='point_id', how='outer', suffixes=('_2018', '_2021'))
df_combined = pd.merge(df_merged_2018_2021, df_2024, on='point_id', how='outer', suffixes=('_2021', '_2024')) # Suffixes apply to the merge with df_2024


print("Combined DataFrame shape (after merging):", df_combined.shape)

# Debugging: Check for point_ids with data from all three years after merging
# A point_id should have columns with _2018, _2021, and no suffix (for 2024)
# Let's check the count of non-null embedding columns for a few point_ids
print("\nDebugging: Checking for data presence across years for first 10 point IDs after merging:")
band_names = ['A' + str(i).zfill(2) for i in range(64)]
band_columns_2018 = [f'{band}_2018' for band in band_names]
band_columns_2021 = [f'{band}_2021' for band in band_names]
band_columns_2024 = band_names # 2024 columns don't have a suffix from the last merge

all_embedding_columns = band_columns_2018 + band_columns_2021 + band_columns_2024

for point_id in df_combined['point_id'].unique()[:10]:
    point_data = df_combined[df_combined['point_id'] == point_id]
    # Check number of non-null embedding columns for each year
    count_2018 = point_data[band_columns_2018].notna().all(axis=1).sum() if all(col in point_data.columns for col in band_columns_2018) else 0
    count_2021 = point_data[band_columns_2021].notna().all(axis=1).sum() if all(col in point_data.columns for col in band_columns_2021) else 0
    count_2024 = point_data[band_columns_2024].notna().all(axis=1).sum() if all(col in point_data.columns for col in band_columns_2024) else 0

    print(f"Point ID {point_id}: 2018 data present = {count_2018 > 0}, 2021 data present = {count_2021 > 0}, 2024 data present = {count_2024 > 0}")


# The subsequent code for splitting data into training and testing sets
# is not relevant to the current task of analyzing embedding changes over time.
# However, to maintain the original notebook structure and avoid errors in subsequent cells,
# I will keep the code for splitting but acknowledge it's not used for the embedding change analysis.

# Separate features (X) and labels (y) - Note: This uses the merged data structure
# Features are the embedding bands (A00 to A63) with year suffixes
# Labels are the 'label' column (from 2018 data)

# Identify the columns that contain the embedding bands for each year
embedding_columns_2018 = [col for col in df_combined.columns if col.startswith('A') and col.endswith('_2018')]
embedding_columns_2021 = [col for col in df_combined.columns if col.startswith('A') and col.endswith('_2021')]
embedding_columns_2024 = [col for col in df_combined.columns if col.startswith('A') and not col.endswith('_2018') and not col.endswith('_2021')] # Assuming 2024 has no suffix from the last merge

# Combine all embedding columns for splitting
all_embedding_columns = embedding_columns_2018 + embedding_columns_2021 + embedding_columns_2024

# Drop rows with any missing embedding data before splitting for training
df_combined_filtered = df_combined.dropna(subset=all_embedding_columns)


X = df_combined_filtered[all_embedding_columns]
y = df_combined_filtered['label_2018'] # Use the label from the 2018 data as the ground truth

print("\nFeatures shape (X):", X.shape)
print("Labels shape (y):", y.shape)

# Handle any potential NaN values in features (should be handled by dropna)
print("\nChecking for NaN values in features (after dropna):")
print(X.isnull().sum().sum()) # Sum of all NaN counts across all columns


# Split the data into training and testing sets
# Using a standard split, e.g., 80% for training and 20% for testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,  # 20% for testing
    random_state=42, # for reproducibility
    stratify=y       # Stratify to maintain the same proportion of labels in train and test sets
)

print("\nData split into training and testing sets:")
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

print("\nDistribution of labels in original data (after filtering):")
print(y.value_counts())
print("\nDistribution of labels in training data:")
print(y_train.value_counts())
print("\nDistribution of labels in testing data:")
print(y_test.value_counts())

Combined DataFrame shape (after merging): (8328, 202)

Debugging: Checking for data presence across years for first 10 point IDs after merging:
Point ID 0: 2018 data present = True, 2021 data present = True, 2024 data present = True
Point ID 1: 2018 data present = True, 2021 data present = True, 2024 data present = True
Point ID 2: 2018 data present = True, 2021 data present = True, 2024 data present = True
Point ID 3: 2018 data present = True, 2021 data present = True, 2024 data present = True
Point ID 4: 2018 data present = True, 2021 data present = True, 2024 data present = True
Point ID 5: 2018 data present = True, 2021 data present = True, 2024 data present = True
Point ID 6: 2018 data present = True, 2021 data present = True, 2024 data present = True
Point ID 7: 2018 data present = True, 2021 data present = True, 2024 data present = True
Point ID 8: 2018 data present = True, 2021 data present = True, 2024 data present = True
Point ID 9: 2018 data present = True, 2021 data present

In [21]:
# Check the frequency of each point_id in the combined dataframe
point_id_counts = df_combined['point_id'].value_counts()

print("Frequency of each point_id in the combined dataframe:")
print(point_id_counts.head()) # Display head for brevity

# Check if any point_id appears less than 3 times (for years 2018, 2021, 2024)
points_with_missing_years = point_id_counts[point_id_counts < 3]

if not points_with_missing_years.empty:
    print("\nPoint IDs with data missing for one or more years:")
    print(points_with_missing_years)
    print(f"\nTotal number of points with missing data: {len(points_with_missing_years)}")
else:
    print("\nAll unique point IDs have data for all three years (2018, 2021, 2024).")

Frequency of each point_id in the combined dataframe:
point_id
8327    1
0       1
1       1
2       1
3       1
Name: count, dtype: int64

Point IDs with data missing for one or more years:
point_id
8327    1
0       1
1       1
2       1
3       1
       ..
12      1
11      1
10      1
9       1
8       1
Name: count, Length: 8328, dtype: int64

Total number of points with missing data: 8328


In [24]:
# Separate the DataFrame into mining (label=1) and non-mining (label=0) sites

df_mining = df_embedding_changes[df_embedding_changes['label'] == 1]
df_non_mining = df_embedding_changes[df_embedding_changes['label'] == 0]

print("Analyzing embedding changes for Mining Sites (label=1):")
# Calculate descriptive statistics for embedding change metrics for mining sites
mining_change_stats = df_mining[['diff_2018_2021', 'diff_2021_2024', 'diff_2018_2024', 'sim_2018_2021', 'sim_2021_2024', 'sim_2018_2024']].describe()
display(mining_change_stats)

print("\nAnalyzing embedding changes for Non-Mining Samples (label=0):")
# Calculate descriptive statistics for embedding change metrics for non-mining sites
non_mining_change_stats = df_non_mining[['diff_2018_2021', 'diff_2021_2024', 'diff_2018_2024', 'sim_2018_2021', 'sim_2021_2024', 'sim_2018_2024']].describe()
display(non_mining_change_stats)

Analyzing embedding changes for Mining Sites (label=1):


,diff_2018_2021,diff_2021_2024,diff_2018_2024,sim_2018_2021,sim_2021_2024,sim_2018_2024
count,6062.000000,6062.000000,6062.000000,6062.000000,6062.000000,6062.000000
mean,0.256906,0.263359,0.295821,0.962197,0.959510,0.949325
std,0.098108,0.107927,0.117748,0.036946,0.042064,0.049252
min,0.071688,0.067271,0.097306,0.336117,0.084304,0.097191
25%,0.191915,0.189725,0.214734,0.956134,0.951203,0.938878
50%,0.236781,0.238707,0.271610,0.971962,0.971484,0.963206
75%,0.296258,0.312281,0.349696,0.981603,0.982024,0.976954
max,1.152599,1.350933,1.341439,0.997419,0.997744,0.995255



Analyzing embedding changes for Non-Mining Samples (label=0):


,diff_2018_2021,diff_2021_2024,diff_2018_2024,sim_2018_2021,sim_2021_2024,sim_2018_2024
count,2262.000000,2262.000000,2262.000000,2262.000000,2262.000000,2262.000000
mean,1.112953,0.280109,1.115409,0.273683,0.950908,0.274171
std,0.462904,0.140565,0.455781,0.392022,0.062696,0.389189
min,0.069685,0.063349,0.066218,-0.225040,0.186043,-0.217158
25%,0.912048,0.183106,0.934427,-0.000873,0.942668,0.000518
50%,1.365961,0.252291,1.363539,0.066330,0.968188,0.070303
75%,1.415139,0.338038,1.413823,0.583087,0.983223,0.562980
max,1.565266,1.276421,1.558489,0.997558,0.997992,0.997814


In [23]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd # Ensure pandas is imported if not already

# We have already loaded the exported data into pandas DataFrames:
# df_2018, df_2021, df_2024.
# These DataFrames contain the embeddings for the original training points for each year.
# We have also merged and filtered this data in df_combined_filtered.

print("Starting analysis of embedding changes at known sites using filtered dataframes...")

# df_combined_filtered contains points with complete embedding data for all three years.
# We can directly iterate through its rows to calculate the embedding change metrics.

embedding_changes = []

print(f"Analyzing embedding changes for {df_combined_filtered.shape[0]} filtered points...")

# Band names were defined previously, ensure they are available or redefine if necessary
# Assuming band_names is already defined from the export configuration step:
band_names = ['A' + str(i).zfill(2) for i in range(64)]

# Construct the full list of embedding band columns for each year after merging
band_columns_2018 = [f'{band}_2018' for band in band_names]
band_columns_2021 = [f'{band}_2021' for band in band_names]
band_columns_2024 = band_names # 2024 columns don't have a suffix from the last merge


# Iterate through each row in the filtered combined dataframe
for index, row in df_combined_filtered.iterrows():
    point_id = row['point_id']
    label = row['label_2018']
    location = row['location_2018'] # Use the location from the 2018 data

    # Extract embeddings for each year
    # Since we filtered for NaNs, these should be complete
    emb_2018 = row[band_columns_2018].values.astype(float)
    emb_2021 = row[band_columns_2021].values.astype(float)
    emb_2024 = row[band_columns_2024].values.astype(float)

    # Calculate difference (e.g., L2 norm of the difference vector)
    diff_2018_2021 = np.linalg.norm(emb_2021 - emb_2018)
    diff_2021_2024 = np.linalg.norm(emb_2024 - emb_2021)
    diff_2018_2024 = np.linalg.norm(emb_2024 - emb_2018)

    # Calculate cosine similarity
    # Reshape for cosine similarity (requires 2D arrays)
    emb_2018_2d = emb_2018.reshape(1, -1)
    emb_2021_2d = emb_2021.reshape(1, -1)
    emb_2024_2d = emb_2024.reshape(1, -1)

    sim_2018_2021 = cosine_similarity(emb_2018_2d, emb_2021_2d)[0][0]
    sim_2021_2024 = cosine_similarity(emb_2021_2d, emb_2024_2d)[0][0]
    sim_2018_2024 = cosine_similarity(emb_2018_2d, emb_2024_2d)[0][0]

    embedding_changes.append({
        'point_id': point_id,
        'label': label,
        'location': location,
        'diff_2018_2021': diff_2018_2021,
        'diff_2021_2024': diff_2021_2024,
        'diff_2018_2024': diff_2018_2024,
        'sim_2018_2021': sim_2018_2021,
        'sim_2021_2024': sim_2021_2024,
        'sim_2018_2024': sim_2018_2024
    })


# Create a DataFrame from the collected change metrics
df_embedding_changes = pd.DataFrame(embedding_changes)

print("\nEmbedding change metrics calculated.")
print("DataFrame shape:", df_embedding_changes.shape)
print("\nFirst 5 rows of embedding change DataFrame:")
display(df_embedding_changes.head())

# Now you can analyze df_embedding_changes. For example, compare metrics for mining vs non-mining sites,
# or analyze changes in specific locations/countries.

Starting analysis of embedding changes at known sites using filtered dataframes...
Analyzing embedding changes for 8324 filtered points...

Embedding change metrics calculated.
DataFrame shape: (8324, 9)

First 5 rows of embedding change DataFrame:


,point_id,label,location,diff_2018_2021,diff_2021_2024,diff_2018_2024,sim_2018_2021,sim_2021_2024,sim_2018_2024
0,0,1.0,NaN,0.136722,0.249703,0.248329,0.990674,0.968817,0.969219
1,1,1.0,NaN,0.164686,0.221319,0.258633,0.986412,0.975407,0.966545
2,2,1.0,NaN,0.147765,0.130601,0.152048,0.989070,0.991484,0.988446
3,3,1.0,NaN,0.164709,0.192887,0.197883,0.986398,0.981288,0.980327
4,4,1.0,NaN,0.505195,0.240269,0.511817,0.872798,0.971213,0.869036


In [25]:
# Visualize Global Cosine Similarity Changes

# To visualize on a map, we need the geographic coordinates (latitude and longitude)
# These are available in the original miningLocations and negativeLocations FeatureCollections.
# We need to retrieve these coordinates and join them with the df_embedding_changes DataFrame.

print("Retrieving geographic coordinates for visualization...")

# Assuming miningLocations and negativeLocations FeatureCollections are loaded in previous cells.

# Convert FeatureCollections to pandas DataFrames to extract coordinates
# This can be done by sampling the geometry information.
# For miningLocations (positive samples):
try:
    # Convert miningLocations to a FeatureCollection with properties including geometry
    mining_points_fc = miningLocations.select(['label', 'location', 'year', '.geo']) # Select relevant properties and geometry
    # Get features as a list of dictionaries
    mining_features_list = mining_points_fc.getInfo()['features']

    # Extract point_id (assuming it corresponds to the original index) and coordinates
    mining_coords = []
    for i, feature in enumerate(mining_features_list):
        # Assuming the original index used for point_id is the order in the FeatureCollection
        point_id = i # Use index as point_id to match the DataFrame
        geometry = feature['geometry']
        if geometry and geometry['type'] == 'Point':
            lon, lat = geometry['coordinates']
            mining_coords.append({'point_id': point_id, 'latitude': lat, 'longitude': lon})

    df_mining_coords = pd.DataFrame(mining_coords)
    print(f"Retrieved coordinates for {len(df_mining_coords)} mining sites.")
    # Debugging: Display head of mining coordinates
    # display(df_mining_coords.head())

except Exception as e:
    print(f"Error retrieving coordinates for mining sites: {e}")
    df_mining_coords = pd.DataFrame() # Create empty DataFrame on error


# For negativeLocations (negative samples):
try:
    # Convert negativeLocations to a FeatureCollection with properties including geometry
    negative_points_fc = negativeLocations.select(['label', 'location', '.geo']) # Select relevant properties and geometry
     # Get features as a list of dictionaries
    negative_features_list = negative_points_fc.getInfo()['features']

    # Extract point_id (assuming it starts after the last mining point_id) and coordinates
    # The negative sample point_ids were added after the mining locations in the original trainingPoints merge.
    # The original index of negativeLocations when merged determined their point_id.
    # To correctly match, we need to know the starting point_id for negative samples.
    # From previous output, numPositive = 6062. So negative sample point_ids start from 6062.
    start_point_id_negative = numPositive # This variable should be available from earlier cells

    negative_coords = []
    for i, feature in enumerate(negative_features_list):
        point_id = start_point_id_negative + i
        geometry = feature['geometry']
        if geometry and geometry['type'] == 'Point':
            lon, lat = geometry['coordinates']
            negative_coords.append({'point_id': point_id, 'latitude': lat, 'longitude': lon})

    df_negative_coords = pd.DataFrame(negative_coords)
    print(f"Retrieved coordinates for {len(df_negative_coords)} negative samples.")
    # Debugging: Display head of negative coordinates
    # display(df_negative_coords.head())

except Exception as e:
    print(f"Error retrieving coordinates for negative samples: {e}")
    df_negative_coords = pd.DataFrame() # Create empty DataFrame on error


# Combine mining and negative coordinates
df_coords = pd.concat([df_mining_coords, df_negative_coords], ignore_index=True)

print(f"\nCombined coordinates DataFrame shape: {df_coords.shape}")
# Debugging: Display head of combined coordinates
# display(df_coords.head())


# Merge the coordinates DataFrame with the df_embedding_changes DataFrame
# We'll use an inner merge to only keep points for which we have both coordinates and embedding changes
df_viz = pd.merge(df_embedding_changes, df_coords, on='point_id', how='inner')

print(f"\nDataFrame for visualization shape (after merging with coordinates): {df_viz.shape}")
# Debugging: Display head of visualization DataFrame
# display(df_viz.head())


# Create a global map

# Choose a cosine similarity metric to visualize (e.g., change from 2018 to 2024)
similarity_metric = 'sim_2018_2024'

# Create a geemap Map object
Map_viz = geemap.Map(center=[0, 0], zoom=2) # Global view

# Add points to the map, colored by the similarity metric
# Normalize the similarity values for coloring (cosine similarity is between -1 and 1)
min_sim = df_viz[similarity_metric].min()
max_sim = df_viz[similarity_metric].max()

# Use a colormap (e.g., 'viridis', 'plasma', 'cividis')
# Viridis goes from purple (low) to yellow (high)
# Let's create a list of points as ee.Feature objects for geemap
features = []
for index, row in df_viz.iterrows():
    point = ee.Geometry.Point(row['longitude'], row['latitude'])
    # Add properties, including the similarity metric
    feature = ee.Feature(point, {
        'point_id': row['point_id'],
        'label': row['label'],
        'location': row['location'],
        similarity_metric: row[similarity_metric] # Include the similarity metric as a property
    })
    features.append(feature)

# Create a FeatureCollection from the list of features
viz_feature_collection = ee.FeatureCollection(features)

# Define visualization parameters for the points based on the similarity metric
# Use a color palette to represent the range of similarity values
palette = ['FF0000', 'FFFF00', '00FF00'] # Red (low sim) -> Yellow (mid sim) -> Green (high sim)
vis_params_points = {
    'palette': palette,
    'min': min_sim, # Use the actual min similarity from data
    'max': max_sim, # Use the actual max similarity from data
    'opacity': 0.8 # Adjust opacity if needed
}

# Add the FeatureCollection to the map
Map_viz.addLayer(viz_feature_collection, vis_params_points, f'Cosine Similarity Change ({similarity_metric})')

# Add a legend
# Create a legend title
legend_title = f'Cosine Similarity ({similarity_metric})'
# Create a legend dictionary
legend_dict = {
    f'{max_sim:.2f} (High Similarity)': palette[2],
    f'{(min_sim + max_sim) / 2:.2f} (Mid Similarity)': palette[1],
    f'{min_sim:.2f} (Low Similarity)': palette[0]
}
Map_viz.add_legend(title=legend_title, legend_dict=legend_dict)


print("\nGlobal map of cosine similarity changes generated.")

# Display the map
Map_viz

Retrieving geographic coordinates for visualization...
Error retrieving coordinates for mining sites: Collection query aborted after accumulating over 5000 elements.
Error retrieving coordinates for negative samples: Collection query aborted after accumulating over 5000 elements.

Combined coordinates DataFrame shape: (0, 0)


KeyError: 'point_id'

In [ ]:
# Investigate changes in specific regions (e.g., China)

# Filter df_embedding_changes for locations within China
# Assuming 'location' column contains country codes or names.
# Need to know the specific values for China in the 'location' column.
# Let's first inspect the unique values in the 'location' column to identify China.


## Analyze embedding changes at known sites

### Subtask:
Analyze the calculated embedding changes (`df_embedding_changes`) by comparing metrics (difference and cosine similarity) between mining and non-mining sites globally.


**Reasoning**:
Separate the DataFrame by label and calculate descriptive statistics for the change metrics for each group.



**Reasoning**:
The KeyError indicates that the 'label' column is missing from the `df_embedding_changes` DataFrame. This is likely because the previous code block where the DataFrame was created failed due to the warnings about missing data for some point IDs. The DataFrame `df_embedding_changes` was printed as empty. Re-create the `df_embedding_changes` DataFrame, handling the missing data gracefully, and then proceed with the analysis.



In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

# Re-create the df_combined DataFrame to ensure it's up-to-date with loaded dataframes
# This was already done in a previous cell, but repeating here for clarity in this step
df_combined = pd.concat([df_2018, df_2021, df_2024], ignore_index=True)

# Get unique point IDs from the combined dataframe
unique_point_ids = df_combined['point_id'].unique()

embedding_changes = []

print(f"Analyzing embedding changes for {len(unique_point_ids)} unique points...")

# Band names were defined previously, ensure they are available or redefine if necessary
# Assuming band_names is already defined from the export configuration step:
band_names = ['A' + str(i).zfill(2) for i in range(64)]


for point_id in unique_point_ids:
    # Get all data for this point across the years
    point_data = df_combined[df_combined['point_id'] == point_id]

    # Check if data for all three years (2018, 2021, 2024) exists for this point_id
    years_present = point_data['year'].unique()
    if 2018 in years_present and 2021 in years_present and 2024 in years_present:

        # Extract embeddings for each year
        emb_2018 = point_data[point_data['year'] == 2018][band_names].values.flatten()
        emb_2021 = point_data[point_data['year'] == 2021][band_names].values.flatten()
        emb_2024 = point_data[point_data['year'] == 2024][band_names].values.flatten()

        # Calculate difference (e.g., L2 norm of the difference vector)
        diff_2018_2021 = np.linalg.norm(emb_2021 - emb_2018)
        diff_2021_2024 = np.linalg.norm(emb_2024 - emb_2021)
        diff_2018_2024 = np.linalg.norm(emb_2024 - emb_2018)

        # Calculate cosine similarity
        # Reshape for cosine similarity (requires 2D arrays)
        emb_2018_2d = emb_2018.reshape(1, -1)
        emb_2021_2d = emb_2021.reshape(1, -1)
        emb_2024_2d = emb_2024.reshape(1, -1)

        sim_2018_2021 = cosine_similarity(emb_2018_2d, emb_2021_2d)[0][0]
        sim_2021_2024 = cosine_similarity(emb_2021_2d, emb_2024_2d)[0][0]
        sim_2018_2024 = cosine_similarity(emb_2018_2d, emb_2024_2d)[0][0]

        # Get the label and location from one of the rows (they should be the same for a given point_id)
        label = point_data['label'].iloc[0]
        location = point_data['location'].iloc[0]

        embedding_changes.append({
            'point_id': point_id,
            'label': label,
            'location': location,
            'diff_2018_2021': diff_2018_2021,
            'diff_2021_2024': diff_2021_2024,
            'diff_2018_2024': diff_2018_2024,
            'sim_2018_2021': sim_2018_2021,
            'sim_2021_2024': sim_2021_2024,
            'sim_2018_2024': sim_2018_2024
        })
    # else:
        # This warning is now less critical as we are explicitly skipping points with incomplete data
        # print(f"Skipping point_id {point_id} due to incomplete data across years.")


# Create a DataFrame from the collected change metrics
df_embedding_changes = pd.DataFrame(embedding_changes)

print("\nEmbedding change metrics calculated.")
print("DataFrame shape:", df_embedding_changes.shape)
print("\nFirst 5 rows of embedding change DataFrame:")
display(df_embedding_changes.head())

# Now proceed with the analysis using the populated df_embedding_changes

print("Starting analysis of embedding changes at known sites using loaded dataframes...")

# Separate the DataFrame into mining (label=1) and non-mining (label=0) sites
df_mining = df_embedding_changes[df_embedding_changes['label'] == 1]
df_non_mining = df_embedding_changes[df_embedding_changes['label'] == 0]

print("\nAnalyzing embedding changes for Mining Sites (label=1):")
# Calculate descriptive statistics for embedding change metrics for mining sites
mining_change_stats = df_mining[['diff_2018_2021', 'diff_2021_2024', 'diff_2018_2024', 'sim_2018_2021', 'sim_2021_2024', 'sim_2018_2024']].describe()
display(mining_change_stats)

print("\nAnalyzing embedding changes for Non-Mining Samples (label=0):")
# Calculate descriptive statistics for embedding change metrics for non-mining sites
non_mining_change_stats = df_non_mining[['diff_2018_2021', 'diff_2021_2024', 'diff_2018_2024', 'sim_2018_2021', 'sim_2021_2024', 'sim_2018_2024']].describe()
display(non_mining_change_stats)

**Reasoning**:
The KeyError persists because `df_embedding_changes` is still empty. This indicates that the condition `if 2018 in years_present and 2021 in years_present and 2024 in years_present:` is filtering out all unique point IDs. This is unexpected given the output from a previous cell showing only 4 points with missing data. There might be an issue with how the years are being checked or how the data was loaded. Let's print the `years_present` for a few point IDs to debug. Also, let's ensure the 'label' column exists in the original dataframes before concatenating.



In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import os # Import os for path joining

# Define the path to your Google Drive folder (ensure this matches the export path)
drive_path = '/content/drive/My Drive/AlphaEarthHack/data/training/'

# Re-load the CSV files into pandas DataFrames to ensure they are correctly loaded
try:
    df_2018 = pd.read_csv(os.path.join(drive_path, 'AlphaEarth_Mining_Training_2018_Mosaic_Revised.csv'))
    df_2021 = pd.read_csv(os.path.join(drive_path, 'AlphaEarth_Mining_Training_2021_Mosaic_Revised.csv'))
    df_2024 = pd.read_csv(os.path.join(drive_path, 'AlphaEarth_Mining_Training_2024_Mosaic_Revised.csv'))

    # Add a 'point_id' column to each dataframe based on the original index if not already present
    if 'point_id' not in df_2018.columns:
        df_2018['point_id'] = df_2018.index
    if 'point_id' not in df_2021.columns:
        df_2021['point_id'] = df_2021.index
    if 'point_id' not in df_2024.columns:
        df_2024['point_id'] = df_2024.index

    print("DataFrames re-loaded successfully.")
    print("df_2018 shape:", df_2018.shape)
    print("df_2021 shape:", df_2021.shape)
    print("df_2024 shape:", df_2024.shape)

    # Check if 'label' column exists in each DataFrame
    print("\nChecking for 'label' column in re-loaded DataFrames:")
    print("df_2018 has 'label':", 'label' in df_2018.columns)
    print("df_2021 has 'label':", 'label' in df_2021.columns)
    print("df_2024 has 'label':", 'label' in df_2024.columns)


except FileNotFoundError as e:
    print(f"Error loading data files: {e}. Please ensure the files exist in the specified Google Drive path.")
    # Exit the subtask if files are not found
    # The subtask cannot be completed without the dataframes.
    raise e # Re-raise the exception to stop execution


# Combine the data from all three years
df_combined = pd.concat([df_2018, df_2021, df_2024], ignore_index=True)

print("\nCombined DataFrame shape:", df_combined.shape)


# Get unique point IDs from the combined dataframe
unique_point_ids = df_combined['point_id'].unique()

embedding_changes = []

print(f"\nAnalyzing embedding changes for {len(unique_point_ids)} unique points...")

# Band names were defined previously, ensure they are available or redefine if necessary
band_names = ['A' + str(i).zfill(2) for i in range(64)]


# Debugging: Print years present for a few point IDs
print("\nDebugging: Years present for first 10 unique point IDs:")
for i, point_id in enumerate(unique_point_ids[:10]):
    point_data = df_combined[df_combined['point_id'] == point_id]
    years_present = point_data['year'].unique()
    print(f"Point ID {point_id}: Years present = {years_present}")


# Re-populate embedding_changes, skipping points with incomplete data as intended
for point_id in unique_point_ids:
    # Get all data for this point across the years
    point_data = df_combined[df_combined['point_id'] == point_id]

    # Check if data for all three years (2018, 2021, 2024) exists for this point_id
    years_present = point_data['year'].unique()
    if 2018 in years_present and 2021 in years_present and 2024 in years_present:

        # Extract embeddings for each year
        emb_2018 = point_data[point_data['year'] == 2018][band_names].values.flatten()
        emb_2021 = point_data[point_data['year'] == 2021][band_names].values.flatten()
        emb_2024 = point_data[point_data['year'] == 2024][band_names].values.flatten()

        # Calculate difference (e.g., L2 norm of the difference vector)
        diff_2018_2021 = np.linalg.norm(emb_2021 - emb_2018)
        diff_2021_2024 = np.linalg.norm(emb_2024 - emb_2021)
        diff_2018_2024 = np.linalg.norm(emb_2024 - emb_2018)

        # Calculate cosine similarity
        # Reshape for cosine similarity (requires 2D arrays)
        emb_2018_2d = emb_2018.reshape(1, -1)
        emb_2021_2d = emb_2021.reshape(1, -1)
        emb_2024_2d = emb_2024.reshape(1, -1)

        sim_2018_2021 = cosine_similarity(emb_2018_2d, emb_2021_2d)[0][0]
        sim_2021_2024 = cosine_similarity(emb_2021_2d, emb_2024_2d)[0][0]
        sim_2018_2024 = cosine_similarity(emb_2018_2d, emb_2024_2d)[0][0]

        # Get the label and location from one of the rows (they should be the same for a given point_id)
        label = point_data['label'].iloc[0]
        location = point_data['location'].iloc[0]

        embedding_changes.append({
            'point_id': point_id,
            'label': label,
            'location': location,
            'diff_2018_2021': diff_2018_2021,
            'diff_2021_2024': diff_2021_2024,
            'diff_2018_2024': diff_2018_2024,
            'sim_2018_2021': sim_2018_2021,
            'sim_2021_2024': sim_2021_2024,
            'sim_2018_2024': sim_2018_2024
        })


# Create a DataFrame from the collected change metrics
df_embedding_changes = pd.DataFrame(embedding_changes)

print("\nEmbedding change metrics calculated.")
print("DataFrame shape:", df_embedding_changes.shape)
print("\nFirst 5 rows of embedding change DataFrame:")
display(df_embedding_changes.head())

# Now proceed with the analysis using the populated df_embedding_changes

print("\nStarting analysis of embedding changes at known sites using populated dataframe...")

# Separate the DataFrame into mining (label=1) and non-mining (label=0) sites
df_mining = df_embedding_changes[df_embedding_changes['label'] == 1]
df_non_mining = df_embedding_changes[df_embedding_changes['label'] == 0]

print("\nAnalyzing embedding changes for Mining Sites (label=1):")
# Calculate descriptive statistics for embedding change metrics for mining sites
mining_change_stats = df_mining[['diff_2018_2021', 'diff_2021_2024', 'diff_2018_2024', 'sim_2018_2021', 'sim_2021_2024', 'sim_2018_2024']].describe()
display(mining_change_stats)

print("\nAnalyzing embedding changes for Non-Mining Samples (label=0):")
# Calculate descriptive statistics for embedding change metrics for non-mining sites
non_mining_change_stats = df_non_mining[['diff_2018_2021', 'diff_2021_2024', 'diff_2018_2024', 'sim_2018_2021', 'sim_2021_2024', 'sim_2018_2024']].describe()
display(non_mining_change_stats)


**Reasoning**:
The debugging output shows that for the first 10 point IDs, only the year 2018 is present in the combined dataframe. This indicates that the concatenation is not combining the data for the same `point_id` across different years correctly. The `ignore_index=True` in `pd.concat` assigns new indices to the combined dataframe, losing the original index which we were using as `point_id`. To fix this, we need to reset the index *before* concatenating and then use the original index as the `point_id`.



In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import os # Import os for path joining

# Define the path to your Google Drive folder (ensure this matches the export path)
drive_path = '/content/drive/My Drive/AlphaEarthHack/data/training/'

# Re-load the CSV files into pandas DataFrames
try:
    df_2018 = pd.read_csv(os.path.join(drive_path, 'AlphaEarth_Mining_Training_2018_Mosaic_Revised.csv'))
    df_2021 = pd.read_csv(os.path.join(drive_path, 'AlphaEarth_Mining_Training_2021_Mosaic_Revised.csv'))
    df_2024 = pd.read_csv(os.path.join(drive_path, 'AlphaEarth_Mining_Training_2024_Mosaic_Revised.csv'))

    # Add a 'point_id' column to each dataframe based on the original index
    # Do this *before* concatenation and set it as the index temporarily
    df_2018['point_id'] = df_2018.index
    df_2021['point_id'] = df_2021.index
    df_2024['point_id'] = df_2024.index

    print("DataFrames re-loaded and point_id added.")
    print("df_2018 shape:", df_2018.shape)
    print("df_2021 shape:", df_2021.shape)
    print("df_2024 shape:", df_2024.shape)

    # Check if 'label' column exists in each DataFrame
    print("\nChecking for 'label' column in re-loaded DataFrames:")
    print("df_2018 has 'label':", 'label' in df_2018.columns)
    print("df_2021 has 'label':", 'label' in df_2021.columns)
    print("df_2024 has 'label':", 'label' in df_2024.columns)


except FileNotFoundError as e:
    print(f"Error loading data files: {e}. Please ensure the files exist in the specified Google Drive path.")
    # Exit the subtask if files are not found
    raise e # Re-raise the exception to stop execution


# Combine the data from all three years
# Now we don't ignore_index so the original index (which is our point_id) is preserved
df_combined = pd.concat([df_2018, df_2021, df_2024]) # Removed ignore_index=True

print("\nCombined DataFrame shape (after fixing index):", df_combined.shape)

# Get unique point IDs from the combined dataframe
unique_point_ids = df_combined['point_id'].unique()

embedding_changes = []

print(f"\nAnalyzing embedding changes for {len(unique_point_ids)} unique points...")

# Band names were defined previously, ensure they are available or redefine if necessary
band_names = ['A' + str(i).zfill(2) for i in range(64)]


# Debugging: Print years present for a few point IDs after fixing concatenation
print("\nDebugging: Years present for first 10 unique point IDs (after fixing concat):")
for i, point_id in enumerate(unique_point_ids[:10]):
    point_data = df_combined[df_combined['point_id'] == point_id]
    years_present = point_data['year'].unique()
    print(f"Point ID {point_id}: Years present = {years_present}")


# Re-populate embedding_changes, skipping points with incomplete data as intended
for point_id in unique_point_ids:
    # Get all data for this point across the years
    point_data = df_combined[df_combined['point_id'] == point_id]

    # Check if data for all three years (2018, 2021, 2024) exists for this point_id
    years_present = point_data['year'].unique()
    if 2018 in years_present and 2021 in years_present and 2024 in years_present:

        # Extract embeddings for each year
        emb_2018 = point_data[point_data['year'] == 2018][band_names].values.flatten()
        emb_2021 = point_data[point_data['year'] == 2021][band_names].values.flatten()
        emb_2024 = point_data[point_data['year'] == 2024][band_names].values.flatten()

        # Calculate difference (e.g., L2 norm of the difference vector)
        diff_2018_2021 = np.linalg.norm(emb_2021 - emb_2018)
        diff_2021_2024 = np.linalg.norm(emb_2024 - emb_2021)
        diff_2018_2024 = np.linalg.norm(emb_2024 - emb_2018)

        # Calculate cosine similarity
        # Reshape for cosine similarity (requires 2D arrays)
        emb_2018_2d = emb_2018.reshape(1, -1)
        emb_2021_2d = emb_2021.reshape(1, -1)
        emb_2024_2d = emb_2024.reshape(1, -1)

        sim_2018_2021 = cosine_similarity(emb_2018_2d, emb_2021_2d)[0][0]
        sim_2021_2024 = cosine_similarity(emb_2021_2d, emb_2024_2d)[0][0]
        sim_2018_2024 = cosine_similarity(emb_2018_2d, emb_2024_2d)[0][0]

        # Get the label and location from one of the rows (they should be the same for a given point_id)
        label = point_data['label'].iloc[0]
        location = point_data['location'].iloc[0]

        embedding_changes.append({
            'point_id': point_id,
            'label': label,
            'location': location,
            'diff_2018_2021': diff_2018_2021,
            'diff_2021_2024': diff_2021_2024,
            'diff_2018_2024': diff_2018_2024,
            'sim_2018_2021': sim_2018_2021,
            'sim_2021_2024': sim_2021_2024,
            'sim_2018_2024': sim_2018_2024
        })


# Create a DataFrame from the collected change metrics
df_embedding_changes = pd.DataFrame(embedding_changes)

print("\nEmbedding change metrics calculated.")
print("DataFrame shape:", df_embedding_changes.shape)
print("\nFirst 5 rows of embedding change DataFrame:")
display(df_embedding_changes.head())

# Now proceed with the analysis using the populated df_embedding_changes

print("\nStarting analysis of embedding changes at known sites using populated dataframe...")

# Separate the DataFrame into mining (label=1) and non-mining (label=0) sites
df_mining = df_embedding_changes[df_embedding_changes['label'] == 1]
df_non_mining = df_embedding_changes[df_embedding_changes['label'] == 0]

print("\nAnalyzing embedding changes for Mining Sites (label=1):")
# Calculate descriptive statistics for embedding change metrics for mining sites
mining_change_stats = df_mining[['diff_2018_2021', 'diff_2021_2024', 'diff_2018_2024', 'sim_2018_2021', 'sim_2021_2024', 'sim_2018_2024']].describe()
display(mining_change_stats)

print("\nAnalyzing embedding changes for Non-Mining Samples (label=0):")
# Calculate descriptive statistics for embedding change metrics for non-mining sites
non_mining_change_stats = df_non_mining[['diff_2018_2021', 'diff_2021_2024', 'diff_2018_2024', 'sim_2018_2021', 'sim_2021_2024', 'sim_2018_2024']].describe()
display(non_mining_change_stats)

**Reasoning**:
The debugging output still shows that only 2018 data is present for the first 10 point IDs even after attempting to fix the concatenation. This is highly unusual and suggests a deeper issue with how the data from the different CSV files is being loaded or aligned based on `point_id`. The `point_id` was created from the index of each individual year's dataframe. When concatenating, pandas aligns based on the index by default. It appears the indices are not consistent across the years for the same physical point. A more robust approach is needed to ensure that data for the same physical point across different years is correctly grouped. Let's try merging the dataframes on the 'point_id' instead of just concatenating. This will explicitly align the rows based on the 'point_id'.



In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import os # Import os for path joining

# Define the path to your Google Drive folder (ensure this matches the export path)
drive_path = '/content/drive/My Drive/AlphaEarthHack/data/training/'

# Re-load the CSV files into pandas DataFrames
try:
    df_2018 = pd.read_csv(os.path.join(drive_path, 'AlphaEarth_Mining_Training_2018_Mosaic_Revised.csv'))
    df_2021 = pd.read_csv(os.path.join(drive_path, 'AlphaEarth_Mining_Training_2021_Mosaic_Revised.csv'))
    df_2024 = pd.read_csv(os.path.join(drive_path, 'AlphaEarth_Mining_Training_2024_Mosaic_Revised.csv'))

    # Add a 'point_id' column to each dataframe based on the original index
    df_2018['point_id'] = df_2018.index
    df_2021['point_id'] = df_2021.index
    df_2024['point_id'] = df_2024.index

    print("DataFrames re-loaded and point_id added.")
    print("df_2018 shape:", df_2018.shape)
    print("df_2021 shape:", df_2021.shape)
    print("df_2024 shape:", df_2024.shape)

    # Check if 'label' column exists in each DataFrame
    print("\nChecking for 'label' column in re-loaded DataFrames:")
    print("df_2018 has 'label':", 'label' in df_2018.columns)
    print("df_2021 has 'label':", 'label' in df_2021.columns)
    print("df_2024 has 'label':", 'label' in df_2024.columns)


except FileNotFoundError as e:
    print(f"Error loading data files: {e}. Please ensure the files exist in the specified Google Drive path.")
    # Exit the subtask if files are not found
    raise e # Re-raise the exception to stop execution

# Merge the dataframes on 'point_id' to align data for the same point across years
# We'll perform outer merges to keep all points, even if data is missing for a year
df_merged_2018_2021 = pd.merge(df_2018, df_2021, on='point_id', how='outer', suffixes=('_2018', '_2021'))
df_combined = pd.merge(df_merged_2018_2021, df_2024, on='point_id', how='outer', suffixes=('_2021', '_2024')) # Suffixes apply to the merge with df_2024

print("\nCombined DataFrame shape (after merging on point_id):", df_combined.shape)

# Check for columns with different year suffixes after merging
print("\nColumns in the merged DataFrame:")
print(df_combined.columns.tolist())


# Get unique point IDs from the combined dataframe
unique_point_ids = df_combined['point_id'].unique()

embedding_changes = []

print(f"\nAnalyzing embedding changes for {len(unique_point_ids)} unique points...")

# Band names were defined previously, ensure they are available or redefine if necessary
band_names = ['A' + str(i).zfill(2) for i in range(64)]

# Construct the full list of embedding band columns for each year after merging
band_columns_2018 = [f'{band}_2018' for band in band_names]
band_columns_2021 = [f'{band}_2021' for band in band_names]
band_columns_2024 = band_names # 2024 columns don't have a suffix from the last merge


# Iterate through each unique point_id
for point_id in unique_point_ids:
    # Get the row for this point_id from the merged dataframe
    point_data = df_combined[df_combined['point_id'] == point_id].iloc[0] # Get the single row

    # Extract embeddings for each year, checking if the columns exist (i.e., data was present for that year)
    emb_2018 = point_data[band_columns_2018].values.astype(float) if all(col in point_data.index for col in band_columns_2018) else None
    emb_2021 = point_data[band_columns_2021].values.astype(float) if all(col in point_data.index for col in band_columns_2021) else None
    emb_2024 = point_data[band_columns_2024].values.astype(float) if all(col in point_data.index for col in band_columns_2024) else None

    # Proceed only if data for all three years is present for this point_id
    if emb_2018 is not None and emb_2021 is not None and emb_2024 is not None:

        # Calculate difference (e.g., L2 norm of the difference vector)
        diff_2018_2021 = np.linalg.norm(emb_2021 - emb_2018)
        diff_2021_2024 = np.linalg.norm(emb_2024 - emb_2021)
        diff_2018_2024 = np.linalg.norm(emb_2024 - emb_2018)

        # Calculate cosine similarity
        # Reshape for cosine similarity (requires 2D arrays)
        emb_2018_2d = emb_2018.reshape(1, -1)
        emb_2021_2d = emb_2021.reshape(1, -1)
        emb_2024_2d = emb_2024.reshape(1, -1)

        sim_2018_2021 = cosine_similarity(emb_2018_2d, emb_2021_2d)[0][0]
        sim_2021_2024 = cosine_similarity(emb_2021_2d, emb_2024_2d)[0][0]
        sim_2018_2024 = cosine_similarity(emb_2018_2d, emb_2024_2d)[0][0]

        # Get the label and location (from the 2018 columns as they are the original labels)
        label = point_data['label_2018']
        location = point_data['location_2018']


        embedding_changes.append({
            'point_id': point_id,
            'label': label,
            'location': location,
            'diff_2018_2021': diff_2018_2021,
            'diff_2021_2024': diff_2021_2024,
            'diff_2018_2024': diff_2018_2024,
            'sim_2018_2021': sim_2018_2021,
            'sim_2021_2024': sim_2021_2024,
            'sim_2018_2024': sim_2018_2024
        })


# Create a DataFrame from the collected change metrics
df_embedding_changes = pd.DataFrame(embedding_changes)

print("\nEmbedding change metrics calculated.")
print("DataFrame shape:", df_embedding_changes.shape)
print("\nFirst 5 rows of embedding change DataFrame:")
display(df_embedding_changes.head())

# Now proceed with the analysis using the populated df_embedding_changes

print("\nStarting analysis of embedding changes at known sites using populated dataframe...")

# Separate the DataFrame into mining (label=1) and non-mining (label=0) sites
df_mining = df_embedding_changes[df_embedding_changes['label'] == 1]
df_non_mining = df_embedding_changes[df_embedding_changes['label'] == 0]

print("\nAnalyzing embedding changes for Mining Sites (label=1):")
# Calculate descriptive statistics for embedding change metrics for mining sites
mining_change_stats = df_mining[['diff_2018_2021', 'diff_2021_2024', 'diff_2018_2024', 'sim_2018_2021', 'sim_2021_2024', 'sim_2018_2024']].describe()
display(mining_change_stats)

print("\nAnalyzing embedding changes for Non-Mining Samples (label=0):")
# Calculate descriptive statistics for embedding change metrics for non-mining sites
non_mining_change_stats = df_non_mining[['diff_2018_2021', 'diff_2021_2024', 'diff_2018_2024', 'sim_2018_2021', 'sim_2021_2024', 'sim_2018_2024']].describe()
display(non_mining_change_stats)

**Reasoning**:
The ValueError indicates that there are NaN values in the embedding data (the 'A' columns) being passed to `cosine_similarity`. This is happening despite the check `if emb_2018 is not None and emb_2021 is not None and emb_2024 is not None:`, because while the *entire array* might not be None, individual elements within the arrays derived from the merged dataframe can be NaN if data was missing for a specific year for that point. The merge operation with `how='outer'` will introduce NaNs for columns where a point_id existed in one dataframe but not the other. We need to handle these NaNs before calculating differences and similarities. A simple approach is to drop rows from the merged dataframe that have any NaN values in the embedding columns for any of the three years before iterating and calculating the metrics.



In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import os # Import os for path joining

# Define the path to your Google Drive folder (ensure this matches the export path)
drive_path = '/content/drive/My Drive/AlphaEarthHack/data/training/'

# Re-load the CSV files into pandas DataFrames
try:
    df_2018 = pd.read_csv(os.path.join(drive_path, 'AlphaEarth_Mining_Training_2018_Mosaic_Revised.csv'))
    df_2021 = pd.read_csv(os.path.join(drive_path, 'AlphaEarth_Mining_Training_2021_Mosaic_Revised.csv'))
    df_2024 = pd.read_csv(os.path.join(drive_path, 'AlphaEarth_Mining_Training_2024_Mosaic_Revised.csv'))

    # Add a 'point_id' column to each dataframe based on the original index
    df_2018['point_id'] = df_2018.index
    df_2021['point_id'] = df_2021.index
    df_2024['point_id'] = df_2024.index

    print("DataFrames re-loaded and point_id added.")
    print("df_2018 shape:", df_2018.shape)
    print("df_2021 shape:", df_2021.shape)
    print("df_2024 shape:", df_2024.shape)

    # Check if 'label' column exists in each DataFrame
    print("\nChecking for 'label' column in re-loaded DataFrames:")
    print("df_2018 has 'label':", 'label' in df_2018.columns)
    print("df_2021 has 'label':", 'label' in df_2021.columns)
    print("df_2024 has 'label':", 'label' in df_2024.columns)


except FileNotFoundError as e:
    print(f"Error loading data files: {e}. Please ensure the files exist in the specified Google Drive path.")
    # Exit the subtask if files are not found
    raise e # Re-raise the exception to stop execution

# Merge the dataframes on 'point_id' to align data for the same point across years
# We'll perform outer merges to keep all points initially, then filter
df_merged_2018_2021 = pd.merge(df_2018, df_2021, on='point_id', how='outer', suffixes=('_2018', '_2021'))
df_combined = pd.merge(df_merged_2018_2021, df_2024, on='point_id', how='outer', suffixes=('_2021', '_2024')) # Suffixes apply to the merge with df_2024

print("\nCombined DataFrame shape (after merging on point_id):", df_combined.shape)

# Check for columns with different year suffixes after merging
# print("\nColumns in the merged DataFrame:")
# print(df_combined.columns.tolist())


# Band names were defined previously, ensure they are available or redefine if necessary
band_names = ['A' + str(i).zfill(2) for i in range(64)]

# Construct the full list of embedding band columns for each year after merging
band_columns_2018 = [f'{band}_2018' for band in band_names]
band_columns_2021 = [f'{band}_2021' for band in band_names]
band_columns_2024 = band_names # 2024 columns don't have a suffix from the last merge

# Identify all embedding columns across all years
all_embedding_columns = band_columns_2018 + band_columns_2021 + band_columns_2024

# Drop rows where any of the embedding columns for any year have NaN values
df_combined_filtered = df_combined.dropna(subset=all_embedding_columns)

print(f"\nCombined DataFrame shape after dropping rows with missing embedding data: {df_combined_filtered.shape}")

# Get unique point IDs from the filtered combined dataframe
unique_point_ids_filtered = df_combined_filtered['point_id'].unique()

embedding_changes = []

print(f"\nAnalyzing embedding changes for {len(unique_point_ids_filtered)} unique points (after filtering)...")


# Iterate through each unique point_id in the filtered dataframe
for point_id in unique_point_ids_filtered:
    # Get the single row for this point_id from the filtered merged dataframe
    point_data = df_combined_filtered[df_combined_filtered['point_id'] == point_id].iloc[0]

    # Extract embeddings for each year (we know they exist due to filtering)
    emb_2018 = point_data[band_columns_2018].values.astype(float)
    emb_2021 = point_data[band_columns_2021].values.astype(float)
    emb_2024 = point_data[band_columns_2024].values.astype(float)

    # Calculate difference (e.g., L2 norm of the difference vector)
    diff_2018_2021 = np.linalg.norm(emb_2021 - emb_2018)
    diff_2021_2024 = np.linalg.norm(emb_2024 - emb_2021)
    diff_2018_2024 = np.linalg.norm(emb_2024 - emb_2018)

    # Calculate cosine similarity
    # Reshape for cosine similarity (requires 2D arrays)
    emb_2018_2d = emb_2018.reshape(1, -1)
    emb_2021_2d = emb_2021.reshape(1, -1)
    emb_2024_2d = emb_2024.reshape(1, -1)

    sim_2018_2021 = cosine_similarity(emb_2018_2d, emb_2021_2d)[0][0]
    sim_2021_2024 = cosine_similarity(emb_2021_2d, emb_2024_2d)[0][0]
    sim_2018_2024 = cosine_similarity(emb_2018_2d, emb_2024_2d)[0][0]

    # Get the label and location (from the 2018 columns as they are the original labels)
    label = point_data['label_2018']
    location = point_data['location_2018']


    embedding_changes.append({
        'point_id': point_id,
        'label': label,
        'location': location,
        'diff_2018_2021': diff_2018_2021,
        'diff_2021_2024': diff_2021_2024,
        'diff_2018_2024': diff_2018_2024,
        'sim_2018_2021': sim_2018_2021,
        'sim_2021_2024': sim_2021_2024,
        'sim_2018_2024': sim_2018_2024
    })


# Create a DataFrame from the collected change metrics
df_embedding_changes = pd.DataFrame(embedding_changes)

print("\nEmbedding change metrics calculated.")
print("DataFrame shape:", df_embedding_changes.shape)
print("\nFirst 5 rows of embedding change DataFrame:")
display(df_embedding_changes.head())

# Now proceed with the analysis using the populated df_embedding_changes

print("\nStarting analysis of embedding changes at known sites using populated dataframe...")

# Separate the DataFrame into mining (label=1) and non-mining (label=0) sites
df_mining = df_embedding_changes[df_embedding_changes['label'] == 1]
df_non_mining = df_embedding_changes[df_embedding_changes['label'] == 0]

print("\nAnalyzing embedding changes for Mining Sites (label=1):")
# Calculate descriptive statistics for embedding change metrics for mining sites
mining_change_stats = df_mining[['diff_2018_2021', 'diff_2021_2024', 'diff_2018_2024', 'sim_2018_2021', 'sim_2021_2024', 'sim_2018_2024']].describe()
display(mining_change_stats)

print("\nAnalyzing embedding changes for Non-Mining Samples (label=0):")
# Calculate descriptive statistics for embedding change metrics for non-mining sites
non_mining_change_stats = df_non_mining[['diff_2018_2021', 'diff_2021_2024', 'diff_2018_2024', 'sim_2018_2021', 'sim_2021_2024', 'sim_2018_2024']].describe()
display(non_mining_change_stats)